# Session

Most applications follow the same flow:

- Create a session (create_session())
- Pass that session to each run(...)
- Rehydrate by service conversation ID (get_session(...)) or from serialized state

Sessions are agent/service-specific. Reusing a session with a different agent configuration or provider can lead to invalid context. If the serialized session contains a service-side session ID, restore it only for the application user or tenant that owns that ID.

**Note:** When using local sessions, you will need to check the data retention agreement for the service provider to determine if they still store conversation history on their servers (e.g., data at rest)

### Service-Managed ID

Service-managed storage will provide a service-side session ID for retrieving the session information on each call

In [24]:
import importlib
import utils
importlib.reload(utils)

<module 'utils' from 'c:\\Users\\JOSHUAS\\dev\\josh-weston\\microsoft-agent-framework-samples\\storage\\utils.py'>

In [ ]:
agent = utils.new_base_openai_responses_agent(
    name="MyAgent",
    description="A friendly and helpful assistant.",
    instructions="You are a friendly and helpful assistant."
)

session = agent.create_session()

first = await agent.run("My name is Josh", session=session)
second = await agent.run("What is my name?", session=session)

In [16]:
service_session_id = session.service_session_id
session.to_dict()
# session_id is unique and local
# service_session_id starts as None, is the remote service session identifier once assigned if using service-managed history

{'type': 'session',
 'session_id': '34bdaecb-c81a-4750-8282-b92faa9b6335',
 'service_session_id': 'resp_0c953ac78715224a006ae0807bc22c81959764a4abed4aca8d',
 'state': {}}

We take the previous session and give it to a new agent so it can continue the conversation

In [ ]:
other_agent = utils.new_base_openai_responses_agent(
    name="OtherAgent",
    description="Another friendly and helpful assistant.",
    instructions="You are another friendly and helpful assistant."
)
assert service_session_id is not None
session = other_agent.get_session(service_session_id=service_session_id)
response = await other_agent.run("What is my name?", session=session)
print(response.to_dict())

{'type': 'agent_response', 'messages': [{'type': 'message', 'role': 'assistant', 'contents': [{'type': 'text', 'text': 'Your name is Josh.', 'additional_properties': {'openai': {'logprobs': []}, 'logprobs': []}}], 'author_name': 'OtherAgent', 'additional_properties': {}}], 'response_id': 'resp_0c953ac78715224a006ae082ba84a48195b9f32179054953d4', 'created_at': '2026-10-27T10:53:46.000000Z', 'finish_reason': 'stop', 'usage_details': {'reasoning_output_token_count': 0, 'cache_creation_input_token_count': 0, 'openai.cached_input_tokens': 0, 'openai.cache_write_tokens': 0, 'input_token_count': 71, 'openai.reasoning_tokens': 0, 'total_token_count': 86, 'output_token_count': 15, 'cache_read_input_token_count': 0}, 'additional_properties': {}}


In [ ]:
openai_client = utils.new_openai_responses_client()
result = await openai_client.client.responses.retrieve(str(service_session_id))
print(result.output[0].content[0].text) # type: ignore

# The conversation is a linked list, you can traverse it using this ID to get previous responses
# The service does this for you automatically when rebuilding the session. There is no way to retrieve the entire
# conversation history in a single request. You need to traverse the linked list manually if you want previous responses.
print(result.previous_response_id)

Your name is Josh.
resp_0c953ac78715224a006ae0807a6df0819587af12e2f1f3473c


### Local Session



In [ ]:
local_agent = utils.new_base_openai_responses_agent()

# Context

Context providers run around each invocation to add context before execution and process data after execution.
Provides an agent with user preferences, past interactions, or external knowledge

In [ ]:
import asyncio
from typing import Any

from agent_framework import Agent, AgentSession, ContextProvider, SessionContext
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv

load_dotenv()

True